# MedSAE quickstart

This notebook trains a MedSAE, the sparse autoencoder of the ML4H 2026 paper *Explanations that Transfer: Interpreting
Medical Retrieval with Sparse AutoEncoders*, on NFCorpus, compares its retrieval with the dense embeddings, describes
a few of its features and explains one retrieval.

It uses the small encoder `BAAI/bge-small-en-v1.5`, so everything runs on a laptop CPU in a few minutes. The paper
trains on CURE with `BAAI/bge-large-en-v1.5`: see `02_reproduce_cure.py` in the same folder.

MedSAE is not on PyPI yet: install it from a clone of the repository.

```bash
git clone https://github.com/doctolib-lab/medsae.git
pip install -e "medsae/package"
```

In [ ]:
import logging

import pandas as pd
import torch
from IPython.display import Markdown, display

from medsae import Embedder, Interpreter, MedSAE, Trainer, TrainingConfig, evaluate, load_dataset, search

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## 1. Load a dataset

`load_dataset` downloads a benchmark of the paper from the Hugging Face Hub at a pinned revision, with the splits of
the paper, and prints its license and terms of use once. NFCorpus has 3,633 documents and train, val and test
queries.

In [ ]:
dataset = load_dataset("nfcorpus")
print(dataset.describe())
dataset.corpus.head(3)

## 2. Embed it with a frozen encoder

`Embedder` wraps a sentence transformers model. Queries of the BGE v1.5 encoders, the small one used here
included, get the retrieval instruction of their model card, the one the paper used for `BAAI/bge-large-en-v1.5`.
`data.save(...)` stores the arrays so that later runs can reload them with `EmbeddedDataset.load(path, dataset)`.

In [ ]:
embedder = Embedder("BAAI/bge-small-en-v1.5")
data = embedder.embed_dataset(dataset)
data.save("runs/notebook/embeddings")
data.dim

## 3. Create a sparse autoencoder

A TopK SAE with m = 3d features, k of which are active for any text. Pretrained checkpoints, once published, load
with `load_pretrained("bge", k=64)`: one Hub repository per encoder and sparsity k.

In [ ]:
torch.manual_seed(42)  # the initial weights
sae = MedSAE(
    d_in=data.dim,
    n_latents=3 * data.dim,
    k=32,
    embedder=data.embedder,
    query_instruction=data.query_instruction,
)
sae

## 4. Train it

`TrainingConfig()` holds the paper hyperparameters: reconstruction plus InfoNCE loss, Adam with learning rate
5 × 10⁻⁴, batches of 512, 100 epochs and a validation NDCG@10 every 100 steps. The trainer keeps the checkpoint with
the best validation NDCG@10 and saves it in `output_dir`.

In [ ]:
trainer = Trainer(sae, TrainingConfig(), output_dir="runs/notebook/sae").fit(data)
sae = trainer.best_model

## 5. Evaluate retrieval

Every test query is scored against the whole corpus by cosine similarity, and NDCG@10 is computed as in the paper.
Passing `None` instead of an SAE scores the dense embeddings.

In [ ]:
pd.DataFrame(
    {
        "dense embeddings": evaluate(None, data, split="test"),
        f"MedSAE, k = {sae.k}": evaluate(sae, data, split="test"),
    }
).T

## 6. Interpret the features

The `Interpreter` selects features that fire on enough training documents and queries, and builds one card per
feature: top documents and queries, negatives and TF IDF terms. Passing an LLM to `describe` adds a topic, a
description and keywords, for example `OpenAICompatibleLLM("my_model", base_url="http://localhost:8000/v1")` or
`TransformersLLM("google/medgemma-4b-it")`, the model of the paper (gated on the Hub).

In [ ]:
interpreter = Interpreter(sae, data, split="train")
features = interpreter.select_features(n=3)
cards = interpreter.describe(features)
display(Markdown("\n\n".join(card.to_markdown(max_chars=200) for card in cards)))

## 7. Explain one retrieval

The cosine similarity of two sparse codes is a sum of one contribution per feature. `explain` shows the features
whose contributions differ most between two passages, here the passages ranked first and tenth for the first test
query, as in Figure 2 of the paper.

In [ ]:
indices, scores = search(sae, data.queries["test"][:1], data.corpus, top_k=10)
query = dataset.queries["test"].text.iloc[0]
doc_a, doc_b = (dataset.corpus.text.iloc[i] for i in indices[0, [0, 9]])
print(interpreter.explain(query, doc_a, doc_b, cards=cards))

## Next steps

* `03_interpret_features.py` labels the features with an LLM and scores the labels in and out of domain.
* `02_reproduce_cure.py` trains the SAE setting of Tables 2 and 3 on CURE and evaluates it as in Table 1 (GPU
  advised).
* `medsae --help` lists the command line equivalents of every step.